# Music Audio Tagging - Exploration

End-to-end walkthrough of feature extraction, model training, and retrieval.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
from src.features.extractor import AudioFeatureExtractor
from src.models.cnn import CNNModel
from src.models.crnn import CRNNModel
import torch

## Feature Extraction Demo
Generate a synthetic sine wave and extract audio features.

In [ ]:
# synthetic audio for demonstration
sr = 22050
duration = 5.0
t = np.linspace(0, duration, int(sr * duration))
y = np.sin(2 * np.pi * 440 * t).astype(np.float32)

extractor = AudioFeatureExtractor(sample_rate=sr, duration=duration, n_mels=128)

mel = extractor.mel_spectrogram(y)
mfcc = extractor.mfcc(y)
sc = extractor.spectral_contrast(y)
chroma = extractor.chroma(y)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
librosa.display.specshow(mel, sr=sr, hop_length=512, x_axis='time', y_axis='mel', ax=axes[0, 0])
axes[0, 0].set_title('Mel Spectrogram')
librosa.display.specshow(mfcc, sr=sr, hop_length=512, x_axis='time', ax=axes[0, 1])
axes[0, 1].set_title('MFCC')
librosa.display.specshow(sc, sr=sr, hop_length=512, x_axis='time', ax=axes[1, 0])
axes[1, 0].set_title('Spectral Contrast')
librosa.display.specshow(chroma, sr=sr, hop_length=512, x_axis='time', y_axis='chroma', ax=axes[1, 1])
axes[1, 1].set_title('Chroma')
plt.tight_layout()
plt.show()

## Model Architecture Demo

In [ ]:
# forward pass through both architectures
dummy = torch.randn(2, 1, 128, 128)

cnn = CNNModel(num_classes=8)
crnn = CRNNModel(num_classes=8)

with torch.no_grad():
    cnn_out = cnn(dummy)
    crnn_out = crnn(dummy)
    cnn_emb = cnn.get_embedding(dummy)
    crnn_emb = crnn.get_embedding(dummy)

print(f'CNN output: {cnn_out.shape}')
print(f'CRNN output: {crnn_out.shape}')
print(f'CNN embedding: {cnn_emb.shape}')
print(f'CRNN embedding: {crnn_emb.shape}')

## Embedding Similarity Demo

In [ ]:
import numpy as np
from src.inference.similarity import MusicSimilarityIndex

# random embeddings to demonstrate retrieval
rng = np.random.default_rng(42)
embeddings = rng.random((200, 128)).astype(np.float32)
track_ids = list(range(200))

index = MusicSimilarityIndex(embedding_dim=128)
index.build(embeddings, track_ids)

query = embeddings[0]
results = index.search(query, top_k=5)
print('Top 5 similar tracks for track 0:')
for r in results:
    print(f'  track_id={r["track_id"]}, distance={r["distance"]:.4f}')